# **Configuring**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
from transformers import AutoTokenizer

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

In [ ]:
model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


In [ ]:
prompt = "The king walked into the castle"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=50
)

print(tokenizer.decode(outputs[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The king walked into the castle, and the king's face was filled with fear.

"What is this?"

"This is the king's body. It's a piece of gold. It's a piece of gold that was stolen


# **Fine-Tuning**

In [ ]:
import pandas as pd

dataset = pd.read_csv("hf://datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")

In [ ]:
print(dataset.columns)

Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='object')


In [ ]:
def format_row(row):
    return f"User: {row['instruction']}\nFriend: {row['response']}"

dataset["text"] = dataset.apply(format_row, axis=1)

In [ ]:
dataset.head()

,flags,instruction,category,intent,response,text
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...,User: question about cancelling order {{Order ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...,User: i have a question about cancelling oorde...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...,User: i need help cancelling puchase {{Order N...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...,User: I need to cancel purchase {{Order Number...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...,"User: I cannot afford this order, cancel purch..."


In [ ]:
dataset = dataset["text"]

In [ ]:
dataset

Column(["User: question about cancelling order {{Order Number}}\nFriend: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.", "User: i have a question about cancelling oorder {{Order Number}}\nFriend: I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority.", "User: i need help cancelling puchase {{Order Number}}\nFriend: I can sense that you're seeking assistance with canceling your purchase with the purchase number {{Order Number}}. I apologize for any inconvenience caused, and I'm here to guide you through the process.\n\nTo cancel your purchase, please follow these steps:\n\n1. Log into your account: Vis

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

In [ ]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./friend-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Step,Training Loss
10,1.762841
20,1.612667
30,1.670634
40,1.732807
50,1.755826
60,1.726119
70,1.683543
80,1.707975
90,1.681806
100,1.645793


Step,Training Loss
10,1.762841
20,1.612667
30,1.670634
40,1.732807
50,1.755826
60,1.726119
70,1.683543
80,1.707975
90,1.681806
100,1.645793


TrainOutput(global_step=3359, training_loss=1.474737400737466, metrics={'train_runtime': 797.5307, 'train_samples_per_second': 33.694, 'train_steps_per_second': 4.212, 'total_flos': 1761446175178752.0, 'train_loss': 1.474737400737466, 'epoch': 1.0})

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [ ]:
prompt=input("Enter the prompt ")
while(prompt!="_"):
  prompt="User: "+prompt+"\nFriend :"
  inputs = tokenizer(prompt, return_tensors="pt")
  inputs = {k: v.to(device) for k, v in inputs.items()}
  outputs = model.generate(**inputs)
  print(tokenizer.decode(outputs[0]))
  prompt=input("Enter the prompt ")

Enter the prompt hii


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


User: hii
Friend : I'm sorry to hear you that I need to check the status of your order {{Order Number}}
Enter the prompt i need to check the order number


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


User: i need to check the order number
Friend : I'm sorry to hear that you need assistance in checking the order number. I understand that you're
Enter the prompt _
